<a href="https://colab.research.google.com/github/23151651/ISYS2001-23151651/blob/main/A2Explaination3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This is the HARDEST section by far. I've never heard of gradio in my life.
#So I asked Chat GPT: "what do I need in a python code to connect my colab code to gradio"
#And I worked off the response it gave me.

#Gradio upload
import gradio as gr
import pandas as pd

def read_gradio_csv(filepath):
  """Read and clean a CSV file upload through Gradio."""
  if not filepath:
    raise ValueError("Please upload a CSV file first.")
  required = ["Date", "Category", "Description", "Type", "Amount"]
  df = None
  for encoding in ["utf-8", "cp1252", "latin1"]:
    try:
      df = pd.read_csv(filepath, encoding=encoding)
      break
    except UnicodeDecodeError:
      continue
    except pd.errors.ParserError:
      raise ValueError("The CSV format could not be read.")
    if not all(column in df.columns for column in required):
      raise ValueError("The CSV is missing required columns: "+", ".join(required))
    df = df[required].copy()
    df["Amount"] = (df["Amount"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip())
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Type"] = df["Type"].astype(str).str.strip().str.title()
    df["Category"] = df["Category"].astype(str).str.strip()
    df["Description"] = df["Description"].fillna("").astype(str).str.strip()
    invalid = (
        df["Date"].isna()
        | df["Amount"].isna()
        | (df["Amount"] <= 0)
        | (~df["Type"].isin(["Income", "Expense"]))
        | (df["Category"] == ""))
    if invalid.any():
      raise ValueError(f"{int(invalid.sum())} invalid transaction row(s) found. Please correct the dates, amounts, types and categories in your CSV and upload it again.")
      return df

def analyse_for_gradio(source, filepath, manual_rows, monthly_budget):
    """Analyse uploaded or manually entered transactions."""
    try:
        if monthly_budget is None or monthly_budget < 0:
            raise ValueError("Please enter a valid non-negative budget.")
        if source == "Upload CSV":
            df = read_gradio_csv(filepath)
        else:
            if manual_rows is None:
                raise ValueError("Please enter at least one transaction.")
            df = pd.DataFrame(
                manual_rows,
                columns=["Date", "Category", "Description", "Type", "Amount"])
            df = df.dropna(how="all")
            if df.empty:
                raise ValueError("Please enter at least one transaction.")
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df["Category"] = df["Category"].fillna("").astype(str).str.strip()
            df["Description"] = (
                df["Description"].fillna("").astype(str).str.strip())
            df["Type"] = df["Type"].fillna("").astype(str).str.strip().str.title()
            df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
            invalid = (
                df["Date"].isna()
                | df["Amount"].isna()
                | (df["Amount"] <= 0)
                | (~df["Type"].isin(["Income", "Expense"]))
                | (df["Category"] == ""))
            if invalid.any():
                raise ValueError(
                    "Some manual entries are invalid. Check the date, "
                    "category, transaction type and positive amount. "
                    "Use Income or Expense for the type.")
        result = calculate_budget_breakdown(df, float(monthly_budget))
        summary = (
            f"Total income: ${result['total_income']:,.2f}\n"
            f"Total expenses: ${result['total_expenses']:,.2f}\n"
            f"Income minus expenses: ${result['balance']:,.2f}\n"
            f"Monthly budget: ${result['monthly_budget']:,.2f}\n"
            f"Budget remaining: ${result['budget_remaining']:,.2f}\n"
            f"Budget used: {result['budget_used_percentage']:.2f}%")
        if result["spending_by_category"]:
            category_text = "\n".join(
                f"{category}: ${amount:,.2f}"
                for category, amount in sorted(
                    result["spending_by_category"].items(),
                    key=lambda item: item[1],
                    reverse=True))
        else:
            category_text = "No expenses recorded."
        return summary, category_text, df, df
    except (ValueError, TypeError, KeyError) as error:
        return f"Could not analyse transactions: {error}", "", None, None

def chat_about_finances(question, df, monthly_budget):
    """Give Gemini the user's actual financial summary."""
    if df is None:
        return "Please analyse your transactions before asking a question."
    try:
        summary = calculate_budget_breakdown(df, float(monthly_budget))
        return ask_gemini(question, financial_summary=summary)
    except (ValueError, TypeError, KeyError) as error:
        return f"Could not prepare your financial summary: {error}"
with gr.Blocks(title="Personal Finance Assistant") as demo:
    gr.Markdown("# Personal Finance Assistant")
    gr.Markdown(
        "Track your money, analyse your budget and ask Gemini "
        "questions about your financial situation.")
    transaction_state = gr.State(None)
    with gr.Row():
        source = gr.Radio(
            choices=["Manual Entry", "Upload CSV"],
            value="Manual Entry",
            label="How would you like to enter your finances?")
        monthly_budget = gr.Number(
            value=1500,
            minimum=0,
            label="Monthly spending budget ($)")
    gr.Markdown(
        "For manual entry, fill in the table below. "
        "For CSV uploads, use the same column names shown in the table.")
    csv_file = gr.File(
        label="Upload your transactions CSV",
        file_types=[".csv"],
        type="filepath")
    manual_table = gr.Dataframe(
        headers=["Date", "Category", "Description", "Type", "Amount"],
        datatype=["str", "str", "str", "str", "number"],
        value=[
            ["2026-10-01", "Salary", "Monthly pay", "Income", 3000],
            ["2026-10-02", "Rent", "Monthly rent", "Expense", 800],
            ["2026-10-03", "Groceries", "Food shopping", "Expense", 250]],
        row_count=(3, "dynamic"),
        column_count=(5, "fixed"),
        interactive=True,
        label="Transaction entries")
    analyse_button = gr.Button("Analyse My Finances", variant="primary")
    gr.Markdown("## Your financial summary")
    summary_output = gr.Textbox(
        label="Budget and balance",
        lines=6,
        interactive=False)
    category_output = gr.Textbox(
        label="Spending by category",
        lines=5,
        interactive=False)
    history_output = gr.Dataframe(
        label="Transactions analysed",
        interactive=False)
    analyse_button.click(
        fn=analyse_for_gradio,
        inputs=[source, csv_file, manual_table, monthly_budget],
        outputs=[
            summary_output,
            category_output,
            history_output,
            transaction_state])
    gr.Markdown("## Ask your AI finance assistant")
    question_input = gr.Textbox(
        label="Your question",
        placeholder="e.g. How can I reduce my spending?",
        lines=2)
    ask_button = gr.Button("Ask Gemini")
    answer_output = gr.Markdown()
    ask_button.click(
        fn=chat_about_finances,
        inputs=[question_input, transaction_state, monthly_budget],
        outputs=answer_output)
demo.launch()
